In [7]:
import json
from datetime import datetime, timedelta
import pandas as pd
import glob
import os
from sentence_transformers import SentenceTransformer, util
import torch
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import random
import re

/opt/anaconda3/lib/python3.8/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


## Setting Directory

In [8]:
os.chdir("/Users/alexlin/Desktop/R For Substack/news_coverage")

## Importing Training Data with Manual Classes

In [9]:
filepath = f"data/mst/00a2_nyt_articles_sample_annotated.csv"

for_train = pd.read_csv(filepath)

In [10]:
filepath_full = f"data/mst/00_nyt_articles.csv"

nyt_articles = pd.read_csv(filepath_full)

In [11]:
# grabbing the indices of the training df
# because the training sample is unbalanced (too few 1s), we will subset the training sample to only include some of the 0s
for_train_pos = for_train[for_train['poli_con_class'] == 1]
random.seed(123)
for_train_neg = for_train[for_train['poli_con_class'] == 0].sample(n = 100)

for_train_subset = for_train_pos.append(for_train_neg)

In [12]:
for_train_subset_index = for_train_subset['news_index']

## Importing NYT embeddings

In [13]:
nyt_embeddings = np.load(f"data/mst/sentence_embeddings/01_nyt_embeddings.npy")

In [14]:
# making embeddings into a df n articles x n features (384)

nyt_embeddings_df = pd.DataFrame(nyt_embeddings)

nyt_train_embeddings = nyt_embeddings_df.iloc[for_train_subset_index]

In [15]:
# Array of classifications for the training df n articles x 1 

In [16]:
nyt_class = for_train_subset['poli_con_class']

### Combining text embeddings with lexicon score as a separate feature

In [2]:
# importing library for stemming
import nltk
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

nltk.download("punkt")

[nltk_data] Downloading package punkt to /Users/alexlin/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [17]:
## Creating lexicon scores

In [53]:
# importing the nyt to countries build, which has relevant countries, capitals, and named entities
nyt_ner = pd.read_csv("data/mst/04_master_xwalk_nyt_ner.csv")
ner_lexicon = nyt_ner['ner'].to_numpy()

In [ ]:
# Initialize stemmer
stemmer = PorterStemmer()

In [154]:
# first, define a universe of relevant words
poli_conflict_lexicon_raw = {"defense", "ambush", "ammunition", "armed", "arms", "army", "armistice", "battle", "blockade", "bomb", "cannon", "artillery", "civil", 
                           "colonel", "corporal", "conflict", "ceasefire", "corps", "destroy", "destruct", "draft", "fight", "fleet", "fort", "general", 
                           "grenade", "damage", "guerrilla", "gang", "gun", "infantry", "intervention", "legion", "lieutenant", 
                           "military", "militant", "militia", "missile", "munition", "naval", "navy", "patrol", "pentagon", "radar", "rebel", 
                           "regiment", "rifle", "rocket", "sergeant", "shell", "soldier", "submarine", "surrender", "tnt", "troop", "war", "weapon",
                           
                           
                           "adversary", "alliance", "allied", "ally",  "autocrat",
                           "border", "campaign", "combat", "communist", "force", "conspiracy", "dictator", "coalition",
                           "fascist", "humanitarian", "international", "invade", "invasion",
                           "junta", "jurisdiction", "liberate",  "radical", "reactionary", 
                           "revolution", "revolt", "secede", "secession", "treason",
                           
                           "territory", "friction", "brigade", "target", "retaliate", "retaliation", "attack", "casualty", "assault", 
                             "drone", "escalate", "escalation", "p o w", "explode", "explosion", "blast", "kill", "injure", "hostage", "hostages", "junta", "violent", "violence",
                             "occupation", "occupy", "deploy", "fight", "fought", "clash", "clashed", "crisis", "crises", "unrest",
                        
                            "defend", "defense", "famine"}

poli_conflict_lexicon_short = {"defense",  "ammunition", "armed", "army", "armistice", "battle",  "bomb", "artillery",  
                                 "conflict", "cease fire", "destroy", "destruct", "fight",  
                                  "guerrilla", "gang",   
                                 "military", "militant", "militia", "missile", "munition", "rebel", 
                                  "rocket",  "soldier", "troop", "war", "weapon","combat",  "forces", 
                                  "humanitarian",  "invade", "invasion", 
                                 "junta", "territory", "retaliat", "attack", "casualt", "assault", 
                                   "drone", "escalat", "p o w", "explod", "explos",  "kill", "injure", "hostage", 
                                   "violen", 
                                 "deploy", "fight"}

# creating a regex pattern from the lexicon
poli_conflict_patterns = [fr"{word}" for word in poli_conflict_lexicon_raw]

compiled_poli_conflict_patterns = [re.compile(p, re.IGNORECASE) for p in poli_conflict_patterns]

# and then define relevant named entities
# poli_conflict_lexicon = set(stemmer.stem(w) for w in poli_conflict_lexicon_raw)

In [155]:
# then define a function that lightly cleans texts and then generates a lexicon score

def lexicon_score(text, lexicon, stemming = True):
    cleaned_text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    words = word_tokenize(cleaned_text.lower())
    if stemming == True:
        stems = [stemmer.stem(w) for w in words]
    else: 
        stems = words
    return sum(stem in lexicon for stem in stems)
    # return words
    
    
# orrr a function that generates lexicon score using regex instead of simple word matching
def regex_lexicon_score(text, patterns):
    count = 0
    for pat in patterns:
        matches = pat.findall(text)   # all matches of this pattern
        if matches:
            count += len(matches)     # count how many times it appears
    return count


In [156]:
# creating a text column that combines title and abstract

nyt_articles['nyt_txt'] = nyt_articles['nyt_title'] + ". " + nyt_articles['nyt_abstract']
# nyt_articles['nyt_txt'] = nyt_articles['nyt_abstract']

# generating lexicon and NER scores
nyt_articles['lexicon_score'] = nyt_articles['nyt_txt'].astype(str).apply(lambda x: regex_lexicon_score(x, compiled_poli_conflict_patterns))
nyt_articles['ner_lexicon_score'] = nyt_articles['nyt_txt'].astype(str).apply(lambda x: lexicon_score(x, ner_lexicon, stemming = False))

# subsetting to training data
for_train_lexicon_score = nyt_articles[["lexicon_score", "ner_lexicon_score"]].iloc[for_train_subset_index].to_numpy()

In [157]:
# binding lexicon scores to text embeddings
# should give 134 (articles) x 386 (features)


In [158]:
nyt_train = np.hstack((for_train_lexicon_score, nyt_train_embeddings))

nyt_train.shape

(134, 386)

In [159]:
## Training a simple SVM to classify texts relevant to political/conflict 

In [160]:
# importing sklearn library
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.model_selection import cross_validate
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import classification_report
from sklearn.metrics import precision_score, recall_score

In [161]:
# creating a train test split

# X_train, X_test, y_train, y_test = train_test_split(nyt_train, nyt_class, test_size=0.2, random_state=42)

# Set up 5-fold CV

cv = StratifiedKFold(n_splits=8, shuffle=True, random_state=42)

In [171]:
# logistic reg model
nyt_classification_logit = LogisticRegression(max_iter=1000)

# svm model
# reweight = {0: 1, 1: 10} # setting weights to adjust for the fact that there are fewer 1s
nyt_classification_svm = SVC(kernel='linear', probability=True, class_weight= "balanced") 


# fitting model
# nyt_classification_svm.fit(X_train, y_train)


# getting performance metrics
scoring = ['accuracy', 'precision', 'recall', 'f1']

scores = cross_validate(nyt_classification_svm, nyt_train, nyt_class, cv=cv, scoring=scoring) 

In [172]:
scores.get("test_accuracy").mean()
# scores.get("test_precision").mean()

0.9021139705882353

In [173]:
# looking at predicted vs actual 

In [174]:
y_pred = cross_val_predict(nyt_classification_svm, nyt_train, nyt_class, cv=5)
print(classification_report(y_pred, nyt_class))

              precision    recall  f1-score   support

           0       0.92      0.94      0.93        98
           1       0.82      0.78      0.80        36

    accuracy                           0.90       134
   macro avg       0.87      0.86      0.86       134
weighted avg       0.89      0.90      0.89       134



In [153]:
y_pred = cross_val_predict(nyt_classification_logit, nyt_train, nyt_class, cv=3)
for_train_subset['ml_class'] = y_pred
# lexscore = nyt_articles['lexicon_score'].iloc[for_train_subset_index].to_numpy()
# for_train_subset['lexicon_score'] = lexscore
# for_train_subset[for_train_subset['ml_class'] != for_train_subset['poli_con_class']]

"Will ‘Cease-Fire Now’ Drown Out ‘Biden 2024’?"
cleaned_text = re.sub(r'[^a-zA-Z0-9\s]', ' ', "Will ‘Cease-Fire Now’ Drown Out ‘Biden 2024’?")
cleaned_text

word_tokenize(cleaned_text.lower())

['will', 'cease', 'fire', 'now', 'drown', 'out', 'biden', '2024']

## Applying fitted model to all NYT articles

In [108]:
# creating features (11818 arts x 385 features)
nyt_lexicon_score = nyt_articles[['lexicon_score', 'ner_lexicon_score']].to_numpy()
nyt_input = np.hstack([nyt_lexicon_score, nyt_embeddings])

nyt_input.shape

(11818, 386)

In [109]:
# fitting the model
nyt_classification_svm.fit(nyt_train, nyt_class)

# predicting classes for all texts
nyt_predict_classes = nyt_classification_svm.predict(nyt_input)

In [113]:
# exportin 

nyt_articles['ml_predict_classes'] = nyt_predict_classes
nyt_articles.to_csv("data/mst/05_nyt_ml_predict_classes.csv")

## Trying a simple lexicon-based approach instead

In [124]:
# defining eligible lexicon

poli_conflict_lexicon_short = np.array(["defense",  "ammunition", "armed", "army", "armistice", "battle",  "bomb", "artillery",  
                                  "destroy", "destruct", "conflict", "skirmsh" , 
                                 "grenade",  "guerrilla",    
                                 "military", "militant", "militia", "missile", "munition", "rebel", 
                                  "rocket",  "soldier", "troop", "war", "weapon","combat",  
                                  "humanitarian",  "invade", "invasion", 
                                 "junta", "territory", "retaliat", "attack", "casualt", "assault", 
                                   "drone", "escalat", "p o w", "explod", "explos",  "kill", "injure", "hostage", 
                                  "junta", "violen",  "fight", "ceasefire", "cease fire"])

# Join with | to create a regex OR pattern
poli_con_regex = "|".join(poli_conflict_lexicon_short)

In [125]:
# fn to clean string

def remove_special_chars_re(text):
    # This pattern keeps only alphanumeric characters and spaces
    # [^a-zA-Z0-9 ] means "any character NOT in the set of a-z, A-Z, 0-9, or space"
    cleaned_text = re.sub(r'[^a-zA-Z0-9 ]', ' ', text).lower()
    
    return cleaned_text

In [126]:
# creating a cleaned text col 
for_train_subset['cleaned_text'] = for_train_subset['nyt_title'] + "; " + for_train_subset['nyt_abstract']

for_train_subset['cleaned_text'] = for_train_subset['cleaned_text'].apply(lambda x: remove_special_chars_re(x))

In [127]:
for_train_subset['lexicon_based'] = for_train_subset['cleaned_text'].apply(lambda x: 1 if re.search(poli_con_regex, x) else 0)

In [128]:
print(for_train_subset[['lexicon_based', 'poli_con_class']])

     lexicon_based  poli_con_class
1                1               1
2                0               1
6                1               1
30               1               1
58               1               1
..             ...             ...
196              0               0
260              0               0
49               0               0
274              0               0
7                0               0

[130 rows x 2 columns]


In [129]:
precision_lexicon = precision_score(for_train_subset['poli_con_class'], for_train_subset['lexicon_based'])
recall_lexicon = recall_score(for_train_subset['poli_con_class'], for_train_subset['lexicon_based'])

print(f"Precision: {precision_lexicon:.2f}")
print(f"Recall: {recall_lexicon:.2f}")

Precision: 0.62
Recall: 0.93


In [131]:
pd.set_option('display.max_colwidth', None)

print(for_train_subset[(for_train_subset['lexicon_based'] == 1) & (for_train_subset['poli_con_class'] == 0)]['cleaned_text'])

234    takeaways from the hearing in the georgia trump case  fani t  willis  the district attorney  defended her personal conduct in a tense courtroom appearance as defense lawyers sought to disqualify her from the prosecution of donald j  trump and his allies in georgia 
211                                                      a cyberattack on a unitedhealth unit disrupts prescription drug orders  for a week  people have been waylaid at pharmacies after a unit of the nation s largest insurer was shut down by a possible ransomware assault 
12                                                                                 killer mike calls his grammys arrest a  speed bump   the artist was arrested on a misdemeanor battery charge after winning awards for best rap album  best rap performance and best rap song 
9                                                                                                      6 great space images in january  a rocket launching from the ocean  an asteroi